In [1]:
# Imports
import pandas as pd
import numpy as np

In [3]:

# Data Handling
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
column_names = ['id', 'diagnosis'] + [f'feature_{i}' for i in range(1, 31)]
df = pd.read_csv(url, header=None, names=column_names)

# Drop the ID column
df = df.drop('id', axis=1)

# Map diagnosis to binary: Malignant (M) = 1, Benign (B) = 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Separate features (X) and target (y)
X = df.drop('diagnosis', axis=1).values
y = df['diagnosis'].values
X.shape, y.shape

((569, 30), (569,))

In [4]:
# Split into training and test sets
def train_test_split(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    indices = np.random.permutation(len(X))
    test_samples = int(len(X) * test_size)
    
    X_test = X[indices[:test_samples]]
    y_test = y[indices[:test_samples]]
    X_train = X[indices[test_samples:]]
    y_train = y[indices[test_samples:]]
    
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = train_test_split(X, y)

In [5]:
# Gaussian Naive Bayes Algorithm
class GaussianNaiveBayes:
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        # Initialize arrays for mean, variance, and prior probabilities
        self.mean = np.zeros((n_classes, n_features), dtype=np.float64)
        self.var = np.zeros((n_classes, n_features), dtype=np.float64)
        self.priors = np.zeros(n_classes, dtype=np.float64)
        
        # Calculate mean, variance, and prior for each class
        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.mean[idx, :] = X_c.mean(axis=0)
            self.var[idx, :] = X_c.var(axis=0)
            self.priors[idx] = X_c.shape[0] / float(n_samples)
            
    def _calculate_pdf(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx]
        # Gaussian probability density function
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator
        
    def _predict(self, x):
        posteriors = []
        for idx, c in enumerate(self.classes):
            # Using log probabilities to prevent mathematical underflow
            prior = np.log(self.priors[idx])
            class_conditional = np.sum(np.log(self._calculate_pdf(idx, x)))
            posterior = prior + class_conditional
            posteriors.append(posterior)
            
        # Return class with the highest probability
        return self.classes[np.argmax(posteriors)]
        
    def predict(self, X):
        return np.array([self._predict(x) for x in X])


In [6]:
nb = GaussianNaiveBayes()
nb.fit(X_train, y_train)
predictions = nb.predict(X_test)

In [7]:
def accuracy_score(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

acc = accuracy_score(y_test, predictions)
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9646
